# Field Shift — LSTM por zona: máximo 30 km alrededor del agricultor

**Versión por zona** de `hack/colab_rnn.ipynb` (misma arquitectura, mismas fuentes y misma evaluación). Cambios:
1. La región ya no es el recuadro fijo de Amealco: pones el lugar del agricultor (`LAT_AGRICULTOR`, `LON_AGRICULTOR`)
   y `RADIO_KM` (máximo 30). El recuadro de descarga se calcula solo y **el modelo solo usa los píxeles agrícolas
   dentro del círculo**.
2. Nueva entrada: **humedad del suelo en la zona de raíces** (`GWETROOT` de NASA POWER): "qué tan húmeda está la tierra abajo".
3. Los resultados van a `salida/<ZONA_ID>/` (no pisan los de Amealco) e incluyen `zona.json` y la confianza en palabras.
4. Los años malos del SIAP solo se conocen para Amealco: para otra zona esa comparación queda en "sin dato".

Si el círculo cruza el borde de un mosaico MODIS (por ejemplo la latitud 20° N), se bajan dos mosaicos y la descarga
se duplica (~42 GB en lugar de ~21 GB).

---

**Qué hace:** con la temporada del 7 de abril al 27 de julio (7 compuestos de 16 días: NDVI de MODIS, lluvia, temperatura,
radiación y ET₀ de NASA POWER), predice el **NDVI máximo de los compuestos del 28 de julio
al 15 de octubre** de cada píxel agrícola de ~500 m. Así se arma el mapa de "estrés esperado".

**Qué compara (lo pide el reto):** climatología del píxel, regresión Ridge, Random Forest y LSTM.

**Separación sin fuga de datos:** por año. Entrenamiento 2001–2017, validación 2018–2020, prueba 2021–2025.
Normalización y climatología se calculan solo con los años de entrenamiento.

**Solo datos de la pestaña Resources del reto:** MODIS (MOD13A1 y MCD12Q1, archivo LP DAAC), SRTM (SRTMGL1, LP DAAC)
y NASA POWER. Todo corre aquí, en Colab: baja los gránulos originales de Earthdata (3 carriles en paralelo),
los recorta a la región, guarda el recorte en tu Drive, entrena y exporta el modelo a **ONNX** para que el HTML
lo ejecute en el navegador (GitHub Pages, sin servidor).

Antes de correr:
1. Cuenta en https://urs.earthdata.nasa.gov (gratis).
2. Credenciales: si ya existe `MyDrive/fieldshift/.earthdata.json` (lo guarda este notebook la primera vez que el inicio
   de sesión funciona), no se pide nada. Si no, en Colab: barra izquierda → icono de la llave (*Secrets*) → *Add new secret*:
   nombre `EARTHDATA_USER` = tu usuario (no el correo) y `EARTHDATA_PASS` = tu contraseña; activa *Notebook access* en los dos.
   Si no los creas, la celda 1b te los pide en un cuadro de texto.
3. La descarga de MODIS es de ~21 GB (338 archivos de ~61 MB que pasan por el disco de Colab y se borran; en Drive quedan pocos MB).
   Para no gastar GPU mientras baja: corre hasta la celda 1b con entorno **CPU**; cuando diga `Datos listos: True`,
   cambia a **T4 GPU** (Entorno de ejecución → Cambiar tipo) y ejecuta todo (la celda 1b ya no baja nada).

**No hay que subir nada a Drive:** al correr la primera celda autorizas el acceso a Drive y el notebook crea
`MyDrive/fieldshift/` solo (recortes en `datos/`, resultados en `salida/`). Al terminar, los archivos de
`salida/` van al repo: `site/modelo/` y `data/processed/`.

In [ ]:
import sys, subprocess, os, json, glob, math, time
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # preventivo: leer NetCDF desde Drive sin bloqueo de archivos
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    # earthaccess: cliente de NASA para Earthdata (inicio de sesión y descarga) · pyhdf: lee los HDF4 de MODIS
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "netCDF4", "onnx", "onnxruntime", "onnxscript",
                    "earthaccess", "pyhdf"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/fieldshift/datos"
    OUT_DIR = "/content/drive/MyDrive/fieldshift/salida"
    TMP_DIR = "/content/tmp_fieldshift"   # disco local de Colab: cada HDF vive aquí solo mientras se recorta
else:  # solo para probar el notebook fuera de Colab; en Colab esta rama no corre
    DATA_DIR = os.environ.get("FS_DATA_DIR", "data/raw/nasa")
    OUT_DIR = os.environ.get("FS_OUT_DIR", "data/processed")
    TMP_DIR = os.environ.get("FS_TMP_DIR", "data/raw/tmp")
for d in (DATA_DIR, OUT_DIR, TMP_DIR):
    os.makedirs(d, exist_ok=True)

import numpy as np
import pandas as pd
import xarray as xr
import requests
import torch
import torch.nn as nn
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE, "| datos:", DATA_DIR)
print("Versiones → numpy", np.__version__, "| pandas", pd.__version__, "| xarray", xr.__version__, "| torch", torch.__version__)

## 1. Configuración (la misma de `hack/config.json`; Amealco es la región de ejemplo)

In [ ]:
# Lugar que puso el agricultor y radio (máximo 30 km): el modelo solo usa los terrenos agrícolas dentro del círculo
LAT_AGRICULTOR, LON_AGRICULTOR = 19.2695, -99.1738
RADIO_KM = 30
NOMBRE_ZONA, ZONA_ID = "Tlalpan, Ciudad de México", "tlalpan"   # ZONA_ID sin espacios ni acentos
assert 0 < RADIO_KM <= 30, "El radio debe ser de 30 km o menos."
_dlat = RADIO_KM / 111.32
_dlon = RADIO_KM / (111.32 * math.cos(math.radians(LAT_AGRICULTOR)))
CONFIG = {
    "bbox": {"lat_min": math.floor((LAT_AGRICULTOR - _dlat) * 100) / 100, "lat_max": math.ceil((LAT_AGRICULTOR + _dlat) * 100) / 100,
             "lon_min": math.floor((LON_AGRICULTOR - _dlon) * 100) / 100, "lon_max": math.ceil((LON_AGRICULTOR + _dlon) * 100) / 100},
    "centro": {"lat": LAT_AGRICULTOR, "lon": LON_AGRICULTOR},
    "fin": "2026-11-01",
    "doy_entrada": list(range(97, 194, 16)),    # 97..193 -> 7 compuestos de 16 días (7 abr - 27 jul)
    "doy_objetivo": list(range(209, 274, 16)),  # 209..273 -> 5 compuestos (28 jul - 15 oct)
    "train": (2001, 2017), "val": (2018, 2020), "test": (2021, 2025),
    # Años malos del maíz (SIAP) del municipio: solo se conocen para Amealco [2005, 2009, 2011, 2014, 2019, 2020].
    # Para otra zona déjalo vacío (la comparación queda "sin dato") o pon los de su municipio.
    "anios_malos_siap": [],
}
USE_SMAP = False     # SMAP no disponible en esta versión (no se descarga): dejar en False
SEMILLAS = [0, 1, 2, 3, 4]
EPOCAS = 80          # las curvas de la 1.ª corrida seguían bajando en la época 40
FECHA_CONSULTA = time.strftime("%Y-%m-%d")
OUT_DIR = os.path.join(OUT_DIR, ZONA_ID)   # resultados de esta zona en su propia carpeta
os.makedirs(OUT_DIR, exist_ok=True)

if USE_SMAP:
    CONFIG.update({"train": (2016, 2020), "val": (2021, 2022), "test": (2023, 2025)})

def en(rango, y):
    return rango[0] <= y <= rango[1]

b = CONFIG["bbox"]
print(f"Zona {NOMBRE_ZONA} · centro ({LAT_AGRICULTOR}, {LON_AGRICULTOR}) · radio {RADIO_KM} km · salida {OUT_DIR}")
print(f"Configuración lista · región {b['lat_min']}–{b['lat_max']} N, {b['lon_min']}–{b['lon_max']} O · "
      f"entrenamiento {CONFIG['train']} · validación {CONFIG['val']} · prueba {CONFIG['test']} · "
      f"SMAP {'sí' if USE_SMAP else 'no'} · {len(SEMILLAS)} semillas, máx. {EPOCAS} épocas")

## 1b. Datos originales de la NASA (LP DAAC / Earthdata): 3 carriles en paralelo → recorte → Drive
Fuentes, todas de la pestaña Resources del reto:
- **MOD13A1 v061** (MODIS, NDVI 500 m cada 16 días + *pixel reliability*). Se eligió en lugar de MOD13Q1 (250 m)
  porque cada archivo pesa 61 MB en vez de 200 MB: el total baja de ~76 GB a **~21 GB**. Para Amealco el único mosaico
  es **h08v06** (el código calcula los mosaicos con la fórmula del grid sinusoidal y solo baja los necesarios).
  Solo los compuestos con DOY 81–273 (13 por año; entradas 97–193, objetivo 209–273, y 81 para el relleno causal)
  de 2001 a 2026: **338 gránulos de ~61 MB** (tamaños verificados con CMR el 14 nov 2026). Cada HDF se baja al disco
  de Colab, se recorta a la región (~3,600 celdas de 0.005°) y se borra: en el disco nunca hay más de 3 a la vez.
  Tiempo: depende de la red de Colab (a 50 MB/s, ~7 min; a 10 MB/s, ~35 min).
- **MCD12Q1 v061** (uso de suelo IGBP `LC_Type1`, 500 m, anual 2001–2025): 25 gránulos, ~0.34 GB.
- **SRTMGL1 v003** (elevación 1″ ≈ 30 m): 2 mosaicos de 1° (N20W100 y N20W101), ~24 MB.

Al terminar quedan en `DATA_DIR` tres NetCDF pequeños (`modis_ndvi_<bbox>.nc`, `mcd12q1_<bbox>.nc`, `srtm_<bbox>.nc`):
si existen, esta celda no baja nada ni pide credenciales. **Si se corta la red o Colab se desconecta:** vuelve a correr
la celda; cada gránulo ya recortado se guardó en `DATA_DIR/partes_<bbox>/` y solo se bajan los que faltan.
Un gránulo que falla no detiene a los demás: al final se listan los fallidos.

In [ ]:
import threading, shutil, zipfile, re
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

SES = requests.Session()   # consultas públicas a CMR, con reintentos ante cortes de red y errores 5xx
SES.mount("https://", HTTPAdapter(max_retries=Retry(total=8, backoff_factor=3,
          status_forcelist=[429, 500, 502, 503, 504], allowed_methods=["GET"])))
CMR = "https://cmr.earthdata.nasa.gov/search/granules.json"
CARRILES = 3                                   # descargas simultáneas
PERMITIR_INCOMPLETO = False                    # True: arma el NetCDF aunque falten gránulos (se avisa cuántos)
SOLO_LO_BAJADO = False                         # emergencia: True arma el NetCDF de NDVI solo con los gránulos ya recortados
PRODUCTO_NDVI = "MOD13A1"                      # MODIS NDVI 500 m (61 MB por archivo); MOD13Q1 250 m pesaba 200 MB (~76 GB en total)
DOY_MODIS = list(range(81, 274, 16))           # 81, 97, ..., 273: 13 compuestos por año (81 solo para el relleno causal)
ANIOS_MODIS = list(range(2001, int(CONFIG["fin"][:4]) + 1))   # 2000 no entra a train/val/test
SDS_NDVI, SDS_REL, SDS_LC = "500m 16 days NDVI", "500m 16 days pixel reliability", "LC_Type1"   # nombres de LP DAAC

TAG = f"{b['lat_min']:.2f}_{b['lat_max']:.2f}_{b['lon_min']:.2f}_{b['lon_max']:.2f}"
RUTA_NDVI = os.path.join(DATA_DIR, f"modis_ndvi_500m_{TAG}.nc")
RUTA_LC = os.path.join(DATA_DIR, f"mcd12q1_{TAG}.nc")
RUTA_DEM = os.path.join(DATA_DIR, f"srtm_{TAG}.nc")
PARTES = os.path.join(DATA_DIR, f"partes_{TAG}")

# Malla geográfica regular sobre el bbox (centros de celda, lat y lon crecientes): es la malla de `ndvi`
PASO = 0.005          # ~500 m, igual que MOD13A1 y MCD12Q1
NLAT = int(round((b["lat_max"] - b["lat_min"]) / PASO))
NLON = int(round((b["lon_max"] - b["lon_min"]) / PASO))
LAT = np.round(b["lat_min"] + (np.arange(NLAT) + 0.5) * PASO, 6)
LON = np.round(b["lon_min"] + (np.arange(NLON) + 0.5) * PASO, 6)
LAT2, LON2 = np.meshgrid(LAT, LON, indexing="ij")

# Grid sinusoidal de MODIS: esfera R = 6371007.181 m, mosaicos de 1111950.5197 m (36 × 18)
R_MODIS, TILE_M = 6371007.181, 1111950.5197
X_MIN, Y_MAX = -math.pi * R_MODIS, math.pi / 2 * R_MODIS

def indice_sinusoidal(n):
    """Para cada celda de la malla: mosaico (h, v) y píxel (fila, col) MODIS que contiene su centro.
    n = 4800 píxeles por mosaico a 250 m (231.656 m) o 2400 a 500 m. Equivale al vecino más cercano."""
    x = R_MODIS * np.radians(LON2) * np.cos(np.radians(LAT2)) - X_MIN
    y = Y_MAX - R_MODIS * np.radians(LAT2)
    h = np.floor(x / TILE_M).astype(int); v = np.floor(y / TILE_M).astype(int)
    px = TILE_M / n
    col = np.clip(np.floor((x - h * TILE_M) / px).astype(int), 0, n - 1)
    fila = np.clip(np.floor((y - v * TILE_M) / px).astype(int), 0, n - 1)
    return h, v, fila, col

IDX500 = indice_sinusoidal(2400)   # MOD13A1 y MCD12Q1: 2400 px de 463.3 m por mosaico; se calcula una sola vez
TILES = sorted(set(zip(IDX500[0].ravel().tolist(), IDX500[1].ravel().tolist())))
print(f"Malla {NLAT} × {NLON} celdas de {PASO}° · mosaicos MODIS:", [f"h{h:02d}v{v:02d}" for h, v in TILES])

def busca(short_name, version, ext, temporal=None):
    """Gránulos de CMR (público, sin credenciales) que tocan el bbox; regresa nombre, URL del archivo y tamaño (MB)."""
    params = {"short_name": short_name, "version": version, "page_size": 2000,
              "bounding_box": f"{b['lon_min']},{b['lat_min']},{b['lon_max']},{b['lat_max']}"}
    if temporal:
        params["temporal"] = temporal
    salida, pag = [], 1
    while True:
        params["page_num"] = pag
        e = SES.get(CMR, params=params, timeout=120).json()["feed"]["entry"]
        for g in e:
            url = next((l["href"] for l in g.get("links", []) if l.get("href", "").startswith("https")
                        and l["href"].endswith(ext)), None)
            if url:
                salida.append({"nombre": g["title"], "url": url, "mb": float(g.get("granule_size") or 0)})
        if len(e) < params["page_size"]:
            return salida
        pag += 1

def info_modis(g):
    """MOD13A1.A2020097.h08v06.061.xxx -> año, DOY y mosaico."""
    p = g["nombre"].split(".")
    g.update(anio=int(p[1][1:5]), doy=int(p[1][5:8]), tile=(int(p[2][1:3]), int(p[2][4:6])))
    return g

_hilo = threading.local()
_CANDADO_HDF = threading.Lock()   # la librería HDF4 no es segura entre hilos: un recorte a la vez
def sesion():
    if getattr(_hilo, "s", None) is None:
        _hilo.s = earthaccess.get_requests_https_session()   # sesión con tu login de Earthdata (una por carril)
    return _hilo.s

def baja(url, intentos=4):
    """Descarga a TMP_DIR con reintentos; un archivo a medias se llama .parcial y nunca cuenta como listo."""
    ruta = os.path.join(TMP_DIR, os.path.basename(url))
    for i in range(1, intentos + 1):
        try:
            with sesion().get(url, stream=True, timeout=(30, 300)) as r:
                if r.status_code in (401, 403):
                    raise PermissionError(f"Earthdata respondió {r.status_code} para {os.path.basename(url)}: "
                                          "revisa usuario/contraseña en urs.earthdata.nasa.gov")
                r.raise_for_status()
                with open(ruta + ".parcial", "wb") as fh:
                    for trozo in r.iter_content(1 << 22):
                        fh.write(trozo)
            os.replace(ruta + ".parcial", ruta)
            return ruta
        except (requests.RequestException, OSError) as e:
            if isinstance(e, PermissionError) or i == intentos:
                raise
            print(f"  corte en {os.path.basename(url)} ({type(e).__name__}); intento {i}/{intentos}, reintento en {20 * i} s",
                  flush=True)
            _hilo.s = None                                     # sesión nueva por si la anterior caducó
            time.sleep(20 * i)

def extrae(ruta, sds, idx, tile):
    """Lee del HDF4 solo la ventana de píxeles que usa la malla y asigna a cada celda su píxel sinusoidal."""
    from pyhdf.SD import SD, SDC
    h, v, fila, col = idx
    sel = (h == tile[0]) & (v == tile[1])
    with open(ruta, "rb") as fh:
        firma = fh.read(4)
    if firma != b"\x0e\x03\x13\x01":                           # firma de HDF4: si no está, bajó otra cosa (p. ej. HTML)
        raise ValueError(f"{os.path.basename(ruta)} no es un HDF4 (¿página de login en lugar del archivo?)")
    f = SD(ruta, SDC.READ)
    try:
        presentes = list(f.datasets().keys())
        faltan = [s for s in sds if s not in presentes]
        if faltan:
            raise KeyError(f"{os.path.basename(ruta)} no trae los SDS {faltan}. SDS presentes: {presentes}")
        r0, r1 = int(fila[sel].min()), int(fila[sel].max()) + 1
        c0, c1 = int(col[sel].min()), int(col[sel].max()) + 1
        out = {"sel": sel}
        for s in sds:
            ventana = np.asarray(f.select(s)[r0:r1, c0:c1])
            m = np.zeros(sel.shape, ventana.dtype)
            m[sel] = ventana[fila[sel] - r0, col[sel] - c0]
            out[s] = m
        return out
    finally:
        f.end()

def procesa(g, sds, idx, carpeta):
    """Un gránulo: descarga → recorte → .npz pequeño en Drive → borra el HDF. Regresa los MB bajados."""
    destino = os.path.join(carpeta, g["nombre"] + ".npz")
    ruta = baja(g["url"])
    try:
        with _CANDADO_HDF:   # leer la ventana tarda menos de 1 s; la descarga sigue en paralelo
            partes = extrae(ruta, sds, idx, g["tile"])
    finally:
        os.remove(ruta)
    with open(destino + ".parcial", "wb") as fh:
        np.savez_compressed(fh, **{k.replace(" ", "_"): v for k, v in partes.items()})
    os.replace(destino + ".parcial", destino)
    return g["mb"]

def descarga_en_carriles(granulos, sds, idx, carpeta, nombre):
    """Baja en CARRILES hilos los gránulos que aún no tienen su .npz. Regresa la lista de fallidos."""
    os.makedirs(carpeta, exist_ok=True)
    pend = [g for g in granulos if not os.path.exists(os.path.join(carpeta, g["nombre"] + ".npz"))]
    pend.sort(key=lambda g: -g.get("anio", 0))   # primero 2026 y los años de prueba/validación: si se corta, faltan los más viejos
    print(f"[{nombre}] {len(granulos)} gránulos · ya recortados {len(granulos) - len(pend)} · por bajar {len(pend)} "
          f"(~{sum(g['mb'] for g in pend) / 1024:.1f} GB)", flush=True)
    fallidos, hechos, mb, t0 = [], 0, 0.0, time.time()
    with ThreadPoolExecutor(CARRILES) as ex:
        futuros = {ex.submit(procesa, g, sds, idx, carpeta): g for g in pend}
        for fu in as_completed(futuros):
            g = futuros[fu]
            try:
                mb += fu.result(); hechos += 1
                if hechos % 10 == 0 or hechos == len(pend):
                    vel = mb / max(time.time() - t0, 1)
                    resto = (sum(x["mb"] for x in pend) - mb) / max(vel, 1e-6) / 60
                    print(f"[{nombre}] {hechos}/{len(pend)} · {vel:.1f} MB/s · faltan ~{resto:.0f} min", flush=True)
            except Exception as e:                  # un gránulo que falla no tumba a los demás
                fallidos.append(g["nombre"])
                print(f"[{nombre}] FALLÓ {g['nombre']}: {type(e).__name__}: {e}", flush=True)
                if isinstance(e, PermissionError):
                    ex.shutdown(wait=False, cancel_futures=True)
                    raise
    return fallidos

def guarda_nc(ds, ruta):
    """Escribe en el disco local y luego copia a Drive (escribir NetCDF directo en Drive a veces falla)."""
    tmp = os.path.join(TMP_DIR, os.path.basename(ruta))
    ds.to_netcdf(tmp)
    shutil.move(tmp, ruta)
    print("guardado", ruta, os.path.getsize(ruta) // 1024, "KB")

def lee_parte(carpeta, g, sds):
    z = np.load(os.path.join(carpeta, g["nombre"] + ".npz"))
    return z["sel"], [z[s.replace(" ", "_")] for s in sds]

def inicia_sesion():
    """Login de Earthdata con earthaccess usando credencial() (archivo privado en Drive → Secret → variable → pregunta)."""
    global earthaccess
    import earthaccess
    usuario = credencial("EARTHDATA_USER")
    clave = credencial("EARTHDATA_PASS", oculta=True)
    os.environ["EARTHDATA_USERNAME"], os.environ["EARTHDATA_PASSWORD"] = usuario, clave   # solo en memoria de esta sesión
    try:
        auth = earthaccess.login(strategy="environment")
    except Exception as e:
        auth = None
        print("earthaccess.login falló:", type(e).__name__, e)
    if not auth or not getattr(auth, "authenticated", False):
        raise RuntimeError("Earthdata rechazó el inicio de sesión. Revisa usuario y contraseña de https://urs.earthdata.nasa.gov "
                           f"(el usuario, no el correo). Si guardaste unos malos, borra {RUTA_CRED} y vuelve a correr.")
    print("Sesión en Earthdata: OK")
    if not os.path.exists(RUTA_CRED):   # se guardan solo si funcionaron: la próxima vez no se piden
        os.makedirs(os.path.dirname(RUTA_CRED), exist_ok=True)
        json.dump({"EARTHDATA_USER": usuario, "EARTHDATA_PASS": clave}, open(RUTA_CRED, "w"))
        print("Credenciales guardadas en tu Drive (", RUTA_CRED, "): ya no se volverán a pedir")

RUTA_CRED = os.path.join(os.path.dirname(DATA_DIR), ".earthdata.json")   # MyDrive/fieldshift/.earthdata.json (privado, fuera del repo)

def credencial(nombre, oculta=False):
    """1) archivo privado en tu Drive → 2) Secret de Colab → 3) variable de entorno → 4) la pide aquí mismo."""
    if os.path.exists(RUTA_CRED):
        guardado = json.load(open(RUTA_CRED)).get(nombre)
        if guardado:
            return guardado
    try:
        from google.colab import userdata
        valor = userdata.get(nombre)
        if valor:
            return valor.strip()
    except Exception as e:   # SecretNotFoundError, NotebookAccessError, TimeoutException o fuera de Colab
        print(f"No encontré el secret {nombre} ({type(e).__name__}).")
    if os.environ.get(nombre):
        return os.environ[nombre].strip()
    import getpass
    return (getpass.getpass if oculta else input)(f"Escribe tu {nombre} de Earthdata: ").strip()

def ya_estan():
    return all(os.path.exists(r) for r in (RUTA_NDVI, RUTA_LC, RUTA_DEM))

if not ya_estan():
    inicia_sesion()
    for f in glob.glob(os.path.join(TMP_DIR, "*")):     # restos de una corrida interrumpida
        os.remove(f) if os.path.isfile(f) else shutil.rmtree(f, ignore_errors=True)
    problemas = []

    # --- SRTMGL1 v003: mosaicos de 1° (archivo .hgt dentro de un .zip; int16 big-endian, 3601 × 3601) ---
    if not os.path.exists(RUTA_DEM):
        piezas = []
        for g in busca("SRTMGL1", "003", ".zip"):
            m = re.match(r"([NS])(\d+)([EW])(\d+)", g["nombre"])
            lat0 = int(m[2]) * (1 if m[1] == "N" else -1); lon0 = int(m[4]) * (1 if m[3] == "E" else -1)
            ruta = baja(g["url"])
            try:
                with zipfile.ZipFile(ruta) as z:
                    hgt = [n for n in z.namelist() if n.lower().endswith(".hgt")]
                    if not hgt:
                        raise ValueError(f"{g['nombre']}: el zip no trae .hgt; contiene {z.namelist()}")
                    raw = np.frombuffer(z.read(hgt[0]), ">i2")
            finally:
                os.remove(ruta)
            n = int(round(math.sqrt(raw.size)))
            a = raw.reshape(n, n)[:-1, :-1].astype("float32")   # sin la última fila/columna: se traslapan con el vecino
            a[a == -32768] = np.nan
            lat = lat0 + 1 - np.arange(n - 1) / (n - 1); lon = lon0 + np.arange(n - 1) / (n - 1)
            da = xr.DataArray(a, coords={"lat": lat, "lon": lon}, dims=("lat", "lon")).sortby("lat")
            da = da.sel(lat=slice(b["lat_min"] - 0.01, b["lat_max"] + 0.01), lon=slice(b["lon_min"] - 0.01, b["lon_max"] + 0.01))
            if da.size:
                piezas.append(da.to_dataset(name="dem"))
            print(f"[srtm] {g['nombre']}: {dict(da.sizes)}")
        if not piezas:
            problemas.append("SRTMGL1: CMR no regresó mosaicos para la región")
        else:
            dem_ds = xr.combine_by_coords(piezas)
            dem_ds.attrs["fuente"] = "SRTMGL1 v003 (LP DAAC), elevación en m, 1 segundo de arco"
            guarda_nc(dem_ds, RUTA_DEM)

    # --- MCD12Q1 v061: LC_Type1 anual a 500 m (2400 px por mosaico) ---
    if not os.path.exists(RUTA_LC):
        lcg = [info_modis(g) for g in busca("MCD12Q1", "061", ".hdf")]
        lcg = [g for g in lcg if g["tile"] in TILES]
        carpeta = os.path.join(PARTES, "mcd12q1")
        fall = descarga_en_carriles(lcg, [SDS_LC], IDX500, carpeta, "mcd12q1")
        listos = [g for g in lcg if g["nombre"] not in fall]
        if fall and not PERMITIR_INCOMPLETO:
            problemas.append(f"MCD12Q1: {len(fall)} gránulos fallidos")
        elif listos:
            anios = sorted({g["anio"] for g in listos})
            lc = np.full((len(anios), NLAT, NLON), 255, "uint8")    # 255 = sin dato (fill de LC_Type1)
            for g in listos:
                sel, (val,) = lee_parte(carpeta, g, [SDS_LC])
                lc[anios.index(g["anio"])][sel] = val[sel]
            ds = xr.Dataset({"LC_Type1": (("time", "lat", "lon"), lc)},
                            coords={"time": pd.to_datetime([f"{a}-01-01" for a in anios]), "lat": LAT, "lon": LON},
                            attrs={"fuente": f"MCD12Q1 v061 LC_Type1 (IGBP), vecino más cercano en la malla de {PASO}°"})
            guarda_nc(ds, RUTA_LC)

    # --- MOD13A1 v061: NDVI + pixel reliability 500 m, DOY 81–273 de 2001–2026 ---
    if not os.path.exists(RUTA_NDVI):
        mg = [info_modis(g) for g in busca(PRODUCTO_NDVI, "061", ".hdf",
                                           temporal=f"{ANIOS_MODIS[0]}-01-01T00:00:00Z,{CONFIG['fin']}T23:59:59Z")]
        mg = [g for g in mg if g["tile"] in TILES and g["doy"] in DOY_MODIS and g["anio"] in ANIOS_MODIS]
        if SOLO_LO_BAJADO:   # no baja nada más; para completar después, borra RUTA_NDVI y vuelve a correr con False
            mg = [g for g in mg if os.path.exists(os.path.join(PARTES, PRODUCTO_NDVI.lower(), g["nombre"] + ".npz"))]
        esperados = len(ANIOS_MODIS) * len(DOY_MODIS) * len(TILES)
        print(f"[{PRODUCTO_NDVI}] en CMR: {len(mg)} de {esperados} esperados ({len(TILES)} mosaico(s) × "
              f"{len(DOY_MODIS)} compuestos × {len(ANIOS_MODIS)} años) · ~{sum(g['mb'] for g in mg) / 1024:.1f} GB")
        carpeta = os.path.join(PARTES, PRODUCTO_NDVI.lower())
        fall = descarga_en_carriles(mg, [SDS_NDVI, SDS_REL], IDX500, carpeta, PRODUCTO_NDVI.lower())
        listos = [g for g in mg if g["nombre"] not in fall]
        if fall and not PERMITIR_INCOMPLETO:
            problemas.append(f"{PRODUCTO_NDVI}: {len(fall)} gránulos fallidos: {fall[:5]}{' …' if len(fall) > 5 else ''}")
        elif listos:
            fechas = sorted({(g["anio"], g["doy"]) for g in listos})
            nd = np.full((len(fechas), NLAT, NLON), np.nan, "float32")
            rl = np.full((len(fechas), NLAT, NLON), -1, "int8")       # -1 = sin dato (fill de pixel reliability)
            for g in listos:
                k = fechas.index((g["anio"], g["doy"]))
                sel, (crudo, rel_) = lee_parte(carpeta, g, [SDS_NDVI, SDS_REL])
                nd[k][sel] = np.where(crudo[sel] == -3000, np.nan, crudo[sel] * 0.0001)   # int16, escala 0.0001, fill -3000
                rl[k][sel] = rel_[sel]
            t = pd.to_datetime([f"{a}-01-01" for a, _ in fechas]) + pd.to_timedelta([d - 1 for _, d in fechas], unit="D")
            ds = xr.Dataset({"ndvi": (("time", "lat", "lon"), nd), "rel": (("time", "lat", "lon"), rl)},
                            coords={"time": t, "lat": LAT, "lon": LON},
                            attrs={"fuente": f"{PRODUCTO_NDVI} v061 (LP DAAC): 500m 16 days NDVI y pixel reliability; "
                                             f"vecino más cercano en la malla de {PASO}°; time = inicio del compuesto"})
            guarda_nc(ds, RUTA_NDVI)

    for p in problemas:
        print("PENDIENTE:", p)

print("Datos listos:", ya_estan())
if not ya_estan():
    raise RuntimeError("Faltan datos (ver arriba). Vuelve a correr esta celda: solo baja los gránulos que faltan; "
                       "las celdas siguientes necesitan los tres NetCDF.")

## 2. MODIS NDVI (MOD13A1, 500 m) + uso de suelo (MCD12Q1)

In [ ]:
def ordena(da):
    da = da.transpose("time", "lat", "lon") if "time" in da.dims else da.transpose("lat", "lon")
    return da.sortby("lat").sortby("lon")

mod = xr.open_dataset(RUTA_NDVI).load()
ndvi = ordena(mod["ndvi"]).astype("float32")      # ya en escala 0–1 (se aplicó 0.0001 al recortar)
rel = ordena(mod["rel"])
ndvi = ndvi.where(ndvi > -0.2)
ndvi = ndvi.where(rel.isin([0, 1]))  # 0 buena, 1 marginal; 2 nieve, 3 nube y -1 sin dato -> hueco
ndvi = ndvi.load()

t = pd.to_datetime(ndvi["time"].values)
ndvi = ndvi.assign_coords(anio=("time", t.year), doy=("time", t.dayofyear))
print("NDVI:", dict(ndvi.sizes), "| huecos:", float(ndvi.isnull().mean()) * 100, "%")

# relleno de huecos SOLO con el pasado (causal): cada hueco toma el último compuesto válido, máx. 2 seguidos (32 días).
# Antes era interpolación lineal sobre toda la serie: el compuesto que empieza el 12 jul se llenaba con el del 28 jul,
# que ya es parte del objetivo (fuga). Las ENTRADAS salen de ndvi_f; el OBJETIVO, de ndvi sin rellenar.
def rellena_pasado(a, max_pasos=2):
    a = a.copy()
    ultimo = a[0].copy()
    edad = np.where(np.isfinite(a[0]), 0, max_pasos + 1)
    for k in range(1, a.shape[0]):
        hueco = ~np.isfinite(a[k])
        edad = np.where(hueco, edad + 1, 0)
        a[k] = np.where(hueco & (edad <= max_pasos), ultimo, a[k])
        ultimo = np.where(hueco, ultimo, a[k])
    return a

ndvi_f = ndvi.copy(data=rellena_pasado(ndvi.values))
print("Huecos tras el relleno causal:", round(float(ndvi_f.isnull().mean()) * 100, 1), "%")

# Máscara agrícola: IGBP 12 (cultivos) o 14 (mosaico cultivo/vegetación natural) en >= 50 % de los años
try:
    lc = ordena(xr.open_dataset(RUTA_LC)["LC_Type1"]).load()
    lc = lc.sel(lat=ndvi.lat.values, lon=ndvi.lon.values, method="nearest")   # misma malla: solo asegura el orden
    frac = lc.isin([12, 14]).mean("time") if "time" in lc.dims else lc.isin([12, 14])
    mascara = (frac >= 0.5).values
    fuente_mascara = "MCD12Q1.061 LC_Type1 (IGBP 12 y 14 en >= 50 % de los años)"
except FileNotFoundError:
    mascara = np.ones(ndvi.shape[1:], bool)
    fuente_mascara = "sin máscara (no se encontró MCD12Q1)"
# Solo los píxeles a RADIO_KM o menos del agricultor (distancia sobre la esfera)
_LA, _LO = np.meshgrid(np.radians(ndvi.lat.values), np.radians(ndvi.lon.values), indexing="ij")
_la0, _lo0 = math.radians(LAT_AGRICULTOR), math.radians(LON_AGRICULTOR)
DIST_KM = 2 * 6371 * np.arcsin(np.sqrt(np.sin((_LA - _la0) / 2) ** 2 + np.cos(_la0) * np.cos(_LA) * np.sin((_LO - _lo0) / 2) ** 2))
DENTRO = DIST_KM <= RADIO_KM
mascara = mascara & DENTRO
fuente_mascara += f"; dentro de {RADIO_KM} km de ({LAT_AGRICULTOR}, {LON_AGRICULTOR})"
if mascara.sum() < 300:
    # respaldo: píxeles con temporada marcada (amplitud de NDVI >= 0.2 y primavera baja)
    clim = ndvi_f.groupby("doy").median("time")
    pico = clim.sel(doy=CONFIG["doy_objetivo"], method="nearest")
    prim = clim.sel(doy=CONFIG["doy_entrada"][:3], method="nearest")
    amp = pico.max("doy") - prim.min("doy")
    mascara = ((amp >= 0.2) & (prim.mean("doy") < 0.45)).values & DENTRO
    fuente_mascara = f"heurística de estacionalidad del NDVI (MCD12Q1 dio < 300 píxeles agrícolas); dentro de {RADIO_KM} km"
print("Píxeles agrícolas:", int(mascara.sum()), "de", mascara.size, "|", fuente_mascara)

## 3. NASA POWER (diario) → ventanas de 16 días alineadas con MODIS
POWER tiene celdas de ~0.5°: el clima es el mismo para toda la región; la diferencia entre píxeles viene del NDVI,
la elevación y la pendiente. ET₀ con Hargreaves (FAO-56, ec. 52) a partir de Tmax, Tmin y la radiación extraterrestre.

In [ ]:
def power_diario(lat, lon, inicio="20000101", fin=CONFIG["fin"].replace("-", "")):
    url = ("https://power.larc.nasa.gov/api/temporal/daily/point"
           "?parameters=PRECTOTCORR,T2M_MAX,T2M_MIN,ALLSKY_SFC_SW_DWN,RH2M,GWETROOT"
           f"&community=AG&longitude={lon}&latitude={lat}&start={inicio}&end={fin}&format=JSON")
    p = SES.get(url, timeout=300).json()["properties"]["parameter"]   # SES: reintentos ante cortes de red
    df = pd.DataFrame(p)
    df.index = pd.to_datetime(df.index, format="%Y%m%d")
    return df.replace(-999.0, np.nan)

def ra_mm(lat_deg, doy):
    """Radiación extraterrestre (FAO-56 ec. 21) en mm/día equivalentes."""
    phi = math.radians(lat_deg)
    dr = 1 + 0.033 * np.cos(2 * np.pi * doy / 365)
    dec = 0.409 * np.sin(2 * np.pi * doy / 365 - 1.39)
    ws = np.arccos(-np.tan(phi) * np.tan(dec))
    ra = 24 * 60 / np.pi * 0.0820 * dr * (ws * np.sin(phi) * np.sin(dec) + np.cos(phi) * np.cos(dec) * np.sin(ws))
    return 0.408 * ra

pw = power_diario(CONFIG["centro"]["lat"], CONFIG["centro"]["lon"])
tmed = (pw.T2M_MAX + pw.T2M_MIN) / 2
pw["ET0"] = 0.0023 * ra_mm(CONFIG["centro"]["lat"], pw.index.dayofyear.values) * (tmed + 17.8) * np.sqrt((pw.T2M_MAX - pw.T2M_MIN).clip(lower=0))
print("POWER:", pw.index.min().date(), "→", pw.index.max().date(), "| días:", len(pw))
pw.to_csv(os.path.join(OUT_DIR, "power_diario.csv"))   # copia exacta del clima que usó el modelo (reproducibilidad)

def ventana(df, anio, doy, dias=16):
    ini = pd.Timestamp(anio, 1, 1) + pd.Timedelta(days=doy - 1)
    return df.loc[ini: ini + pd.Timedelta(days=dias - 1)]

CLIMA_VARS = ["lluvia_mm", "tmax_c", "tmin_c", "rad_mj", "et0_mm", "humedad_raiz"]   # humedad_raiz = GWETROOT (0-1)
def clima_paso(anio, doy):
    w = ventana(pw, anio, doy)
    return [w.PRECTOTCORR.sum(min_count=8), w.T2M_MAX.mean(), w.T2M_MIN.min(), w.ALLSKY_SFC_SW_DWN.mean(), w.ET0.sum(min_count=8), w.GWETROOT.mean()]

# comprobación rápida (en Amealco el estudio previo daba 452 mm en jun-sep 2020)
print("Lluvia jun-sep 2020 (POWER):", round(pw.loc["2020-06-01":"2020-09-30", "PRECTOTCORR"].sum(), 1), "mm")

## 4. SRTM: elevación y pendiente en la malla del NDVI (SMAP no disponible en esta versión)
La pendiente se calcula con el DEM de ~30 m y luego se promedia dentro de cada celda de 0.005° (unos 18 × 18 píxeles SRTM).
SMAP: sus archivos diarios son globales y pesan mucho para dos días de hackathon; esta versión no lo descarga
y `USE_SMAP` debe quedar en `False`.

In [ ]:
smap_paso = {}
if USE_SMAP:
    raise RuntimeError("SMAP no está disponible en esta versión del notebook (no se descarga). Pon USE_SMAP = False en la celda 1.")

def a_malla(campo, lat, lon):
    """Promedio (sin NaN) de un campo fino [lat, lon] dentro de cada celda de la malla de `ndvi`."""
    la, lo = ndvi.lat.values, ndvi.lon.values
    paso_y, paso_x = float(la[1] - la[0]), float(lo[1] - lo[0])
    iy_ = np.floor((lat - (la[0] - paso_y / 2)) / paso_y).astype(int)
    ix_ = np.floor((lon - (lo[0] - paso_x / 2)) / paso_x).astype(int)
    IY, IX = np.meshgrid(iy_, ix_, indexing="ij")
    ok = (IY >= 0) & (IY < len(la)) & (IX >= 0) & (IX < len(lo)) & np.isfinite(campo)
    k = IY[ok] * len(lo) + IX[ok]
    suma = np.bincount(k, campo[ok], len(la) * len(lo)); n = np.bincount(k, minlength=len(la) * len(lo))
    return np.where(n > 0, suma / np.maximum(n, 1), np.nan).reshape(len(la), len(lo)).astype("float32")

try:
    dem = ordena(xr.open_dataset(RUTA_DEM)["dem"]).astype("float32").load()
    lat0 = float(dem.lat.mean())
    dy = abs(float(dem.lat[1] - dem.lat[0])) * 111_320
    dx = abs(float(dem.lon[1] - dem.lon[0])) * 111_320 * math.cos(math.radians(lat0))
    gy, gx = np.gradient(dem.values, dy, dx)
    pend = np.degrees(np.arctan(np.hypot(gx, gy)))
    elev250 = a_malla(dem.values, dem.lat.values, dem.lon.values)
    pend250 = a_malla(pend, dem.lat.values, dem.lon.values)
    print("SRTM en la malla:", elev250.shape, "| elevación", round(float(np.nanmin(elev250))), "–",
          round(float(np.nanmax(elev250))), "m | pendiente media", round(float(np.nanmean(pend250)), 1), "°")
except FileNotFoundError:
    elev250 = np.zeros(ndvi.shape[1:], "float32"); pend250 = np.zeros(ndvi.shape[1:], "float32")
    print("Sin SRTM: elevación y pendiente en 0")

## 5. Armar las secuencias: una fila por píxel-año

In [ ]:
iy, ix = np.where(mascara)
lats = ndvi.lat.values[iy]; lons = ndvi.lon.values[ix]
anios_disp = sorted(set(int(a) for a in ndvi.anio.values))

A_T, D_T = ndvi_f.anio.values, ndvi_f.doy.values
NDVI_ENT, NDVI_OBJ = ndvi_f.values, ndvi.values   # entradas: relleno causal | objetivo: solo observaciones válidas
def compuesto(anio, doy, fuente):
    """Compuesto MODIS de ese año más cercano al día doy (tolerancia ±8 días)."""
    sel = np.where((A_T == anio) & (np.abs(D_T - doy) <= 8))[0]
    if not len(sel):
        return None
    return fuente[sel[np.argmin(np.abs(D_T[sel] - doy))]][iy, ix]

filas = []  # (anio, X_seq [P,T,F], y [P])
for y in anios_disp:
    ent = [compuesto(y, d, NDVI_ENT) for d in CONFIG["doy_entrada"]]
    obj = [compuesto(y, d, NDVI_OBJ) for d in CONFIG["doy_objetivo"]]
    if any(e is None for e in ent):
        continue
    pasos = []
    for d, e in zip(CONFIG["doy_entrada"], ent):
        clima = clima_paso(y, d) + ([smap_paso.get((y, d), np.nan)] if USE_SMAP else [])
        pasos.append(np.column_stack([e] + [np.full(len(iy), v, "float32") for v in clima]))
    Xs = np.stack(pasos, 1).astype("float32")                      # [P, T, F]
    obj = [o for o in obj if o is not None]
    pila = np.stack(obj, 1) if obj else np.full((len(iy), 1), np.nan, "float32")    # [P, compuestos del objetivo]
    n_ok = np.isfinite(pila).sum(1)
    # pico = máximo de los compuestos válidos, solo si ESE píxel tiene >= 3 de 5 (si no, una fecha nublada sería el "pico")
    yv = np.where(n_ok >= 3, np.where(np.isfinite(pila), pila, -np.inf).max(1), np.nan)
    filas.append((y, Xs, yv.astype("float32")))

assert filas, f"Ningún año tiene los 7 compuestos de entrada {CONFIG['doy_entrada']}; DOY presentes: {sorted(set(D_T.tolist()))}"
ANIOS = np.array([f[0] for f in filas])
XSEQ = np.stack([f[1] for f in filas])                             # [A, P, T, F]
Y = np.stack([f[2] for f in filas])                                # [A, P]
FEATS = ["ndvi"] + CLIMA_VARS + (["smap"] if USE_SMAP else [])
print("Años:", ANIOS.min(), "→", ANIOS.max(), "| tensor:", XSEQ.shape, "| variables:", FEATS)
print("Píxel-año con objetivo válido (>= 3 compuestos):", round(float(np.isfinite(Y).mean()) * 100, 1), "%")

tr = np.array([en(CONFIG["train"], a) for a in ANIOS])
va = np.array([en(CONFIG["val"], a) for a in ANIOS])
te = np.array([en(CONFIG["test"], a) for a in ANIOS])

# climatología del píxel: SOLO años de entrenamiento (evita fuga)
CLIM = np.nanmean(Y[tr], 0)                                         # [P]
ESTAT = np.column_stack([CLIM, elev250[iy, ix], pend250[iy, ix]]).astype("float32")  # [P, S]

# normalización con estadísticas de entrenamiento
mu_s = np.nanmean(XSEQ[tr], (0, 1, 2)); sd_s = np.nanstd(XSEQ[tr], (0, 1, 2)) + 1e-6
mu_e = np.nanmean(ESTAT, 0); sd_e = np.nanstd(ESTAT, 0) + 1e-6

def aplana(mask_anios):
    A = mask_anios.sum()
    xs = ((XSEQ[mask_anios] - mu_s) / sd_s).reshape(A * len(iy), len(CONFIG["doy_entrada"]), len(FEATS))
    xe = np.tile((ESTAT - mu_e) / sd_e, (A, 1))
    yv = Y[mask_anios].reshape(-1)
    cl = np.tile(CLIM, A)
    ok = np.isfinite(yv) & np.isfinite(xs).all((1, 2)) & np.isfinite(xe).all(1)
    return xs[ok], xe[ok], yv[ok], cl[ok], ok

Xtr, Etr, ytr, ctr, _ = aplana(tr)
Xva, Eva, yva, cva, _ = aplana(va)
Xte, Ete, yte, cte, ok_te = aplana(te)   # ok_te: para saber el año de cada fila de prueba
print("Filas  train:", len(ytr), " val:", len(yva), " test:", len(yte))

## 6. Métricas y líneas base

In [ ]:
def metricas(y, p, clim):
    rmse = float(np.sqrt(np.mean((y - p) ** 2)))
    r2 = float(1 - np.sum((y - p) ** 2) / np.sum((y - y.mean()) ** 2))
    skill = float(1 - np.sum((y - p) ** 2) / np.sum((y - clim) ** 2))   # > 0 = le gana a la climatología
    return {"rmse": round(rmse, 4), "r2": round(r2, 3), "skill_vs_climatologia": round(skill, 3)}

resultados, PRED_VA, PRED_TE = {}, {}, {}
def registra(nombre, p_va, p_te):
    PRED_VA[nombre], PRED_TE[nombre] = p_va, p_te
    resultados[nombre] = {"val": metricas(yva, p_va, cva), "test": metricas(yte, p_te, cte)}

registra("Climatología del píxel", cva, cte)

plano = lambda X, E: np.column_stack([X.reshape(len(X), -1), E])
# Ridge, RF y las redes aprenden la ANOMALÍA (NDVI - promedio del píxel) y luego se les suma el promedio:
# el nivel de cada píxel ya lo da la climatología; el modelo solo tiene que aprender la desviación del año.
ridge = Ridge(alpha=1.0).fit(plano(Xtr, Etr), ytr - ctr)
registra("Ridge", ridge.predict(plano(Xva, Eva)) + cva, ridge.predict(plano(Xte, Ete)) + cte)

rng = np.random.default_rng(0)
sub = rng.choice(len(ytr), min(150_000, len(ytr)), replace=False)
rf = RandomForestRegressor(n_estimators=200, max_depth=16, min_samples_leaf=5, n_jobs=-1, random_state=0)
rf.fit(plano(Xtr, Etr)[sub], (ytr - ctr)[sub])
registra("Random Forest", rf.predict(plano(Xva, Eva)) + cva, rf.predict(plano(Xte, Ete)) + cte)
pd.DataFrame({k: v["test"] for k, v in resultados.items()}).T

## 7. LSTM (5 semillas → promedio e incertidumbre)

In [ ]:
class RedTemporada(nn.Module):
    def __init__(self, n_feat, n_est, oculto=64, capas=2, celda="lstm"):
        super().__init__()
        rnn = nn.LSTM if celda == "lstm" else nn.GRU
        self.rnn = rnn(n_feat, oculto, num_layers=capas, batch_first=True, dropout=0.2 if capas > 1 else 0.0)
        self.cabeza = nn.Sequential(nn.Linear(oculto + n_est, 64), nn.ReLU(), nn.Dropout(0.1), nn.Linear(64, 1))
    def forward(self, xs, xe):
        _, h = self.rnn(xs)
        h = h[0] if isinstance(h, tuple) else h
        return self.cabeza(torch.cat([h[-1], xe], 1)).squeeze(1)

def tensores(*arrs):
    return [torch.tensor(a, dtype=torch.float32) for a in arrs]

CURVAS = {}   # "celda-semilla" -> [(época, RMSE train, RMSE val)]: para ver sobreajuste o subajuste

def entrena(semilla, celda="lstm", lote=4096, capas=2, oculto=64):
    torch.manual_seed(semilla); np.random.seed(semilla)
    m = RedTemporada(len(FEATS), ESTAT.shape[1], oculto=oculto, capas=capas, celda=celda).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=2e-3, weight_decay=1e-5)
    xs, xe, yy = tensores(Xtr, Etr, ytr - ctr)                       # objetivo = anomalía del píxel
    vs, ve, vy = [t.to(DEVICE) for t in tensores(Xva, Eva, yva - cva)]
    mejor, estado, paciencia = 1e9, None, 0
    curva = CURVAS[f"{celda}-{semilla}"] = []
    for ep in range(EPOCAS):
        m.train()
        perm = torch.randperm(len(yy))
        suma, n = 0.0, 0
        for i in range(0, len(yy), lote):
            j = perm[i:i + lote]
            opt.zero_grad()
            perdida = nn.functional.mse_loss(m(xs[j].to(DEVICE), xe[j].to(DEVICE)), yy[j].to(DEVICE))
            perdida.backward(); opt.step()
            suma += perdida.detach().item() * len(j); n += len(j)
        m.eval()
        with torch.no_grad():
            pv = float(nn.functional.mse_loss(m(vs, ve), vy))
        curva.append((ep + 1, round((suma / n) ** 0.5, 4), round(pv ** 0.5, 4)))
        if pv < mejor - 1e-6:
            mejor, estado, paciencia = pv, {k: v.detach().clone() for k, v in m.state_dict().items()}, 0
        else:
            paciencia += 1
            if paciencia >= 8:
                break
    m.load_state_dict(estado)
    print(f"semilla {semilla}: épocas {ep + 1}, RMSE val {mejor ** 0.5:.4f}")
    return m

def predice(m, X, E, lote=65536):
    m.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(X), lote):
            xs, xe = tensores(X[i:i + lote], E[i:i + lote])
            out.append(m(xs.to(DEVICE), xe.to(DEVICE)).cpu().numpy())
    # la red da la anomalía; se le suma el promedio del píxel (columna 0 de las estáticas, desnormalizada)
    return np.concatenate(out) + (E[:, 0] * sd_e[0] + mu_e[0])

# Dos redes recurrentes con las mismas entradas y semillas: la LSTM 2×64 (~56 mil parámetros) y una GRU 1×32
# (~6 mil). Con solo 17 temporadas de clima, la red chica puede generalizar mejor; se elige con VALIDACIÓN.
FAMILIAS = {"LSTM (5 semillas)": dict(celda="lstm", capas=2, oculto=64),
            "GRU 1×32 (5 semillas)": dict(celda="gru", capas=1, oculto=32)}
REDES = {}
for nombre_red, arq in FAMILIAS.items():
    REDES[nombre_red] = [entrena(s, **arq) for s in SEMILLAS]
    registra(nombre_red, np.stack([predice(m, Xva, Eva) for m in REDES[nombre_red]]).mean(0),
             np.stack([predice(m, Xte, Ete) for m in REDES[nombre_red]]).mean(0))
for clave, c in CURVAS.items():   # (época, RMSE train, RMSE val): si val sube mientras train baja = sobreajuste
    print("curva", clave, c)
RED = min(FAMILIAS, key=lambda k: resultados[k]["val"]["rmse"])   # red del mapa, los escenarios y el ONNX
modelos = REDES[RED]
print("Red elegida con validación para el sitio:", RED, "| RMSE val:", {k: resultados[k]["val"]["rmse"] for k in FAMILIAS})
P_va = np.stack([predice(m, Xva, Eva) for m in modelos]); P_te = np.stack([predice(m, Xte, Ete) for m in modelos])

# Incertidumbre: la dispersión entre 5 semillas no incluye el error del modelo y sale muy angosta.
# Banda = raíz(de_semillas² + sigma_res²), con sigma_res = RMSE de la LSTM en VALIDACIÓN (no en prueba).
sigma_res = float(np.sqrt(np.mean((yva - P_va.mean(0)) ** 2)))
banda_semillas = float(np.mean(P_te.std(0)))
banda_te = np.sqrt(P_te.std(0) ** 2 + sigma_res ** 2)
banda_test = float(np.mean(banda_te))
cob_sem = float(np.mean(np.abs(yte - P_te.mean(0)) <= P_te.std(0)))
cob_tot = float(np.mean(np.abs(yte - P_te.mean(0)) <= banda_te))
print(f"Cobertura de ±1 banda en prueba (ideal ~68 %): solo semillas {cob_sem:.0%} | semillas + sigma_res {cob_tot:.0%}")

tabla = pd.DataFrame({k: v["test"] for k, v in resultados.items()}).T
print(f"Prueba {CONFIG['test'][0]}-{CONFIG['test'][1]} (años nunca vistos):"); print(tabla)

# skill por año de prueba: con 5 años y el mismo clima en todos los píxeles, el promedio puede venir de un solo año
anio_te = np.repeat(ANIOS[te], len(iy))[ok_te]
por_anio = {k: {int(a): round(float(1 - np.sum((yte[anio_te == a] - p[anio_te == a]) ** 2)
                                      / np.sum((yte[anio_te == a] - cte[anio_te == a]) ** 2)), 3)
                for a in np.unique(anio_te)} for k, p in PRED_TE.items()}
print("Skill vs climatología por año de prueba (> 0 = le gana):"); print(pd.DataFrame(por_anio))

tabla_val = pd.DataFrame({k: v["val"] for k, v in resultados.items()}).T
mejor_modelo = tabla_val["rmse"].idxmin()       # se elige con VALIDACIÓN; la prueba solo se reporta
print("Mejor en validación:", mejor_modelo, "| su skill en prueba:", tabla.loc[mejor_modelo, "skill_vs_climatologia"],
      "| banda media LSTM (±1):", round(banda_test, 4), "| solo semillas:", round(banda_semillas, 4))
if mejor_modelo != RED:
    print(f"OJO: el mejor en validación es {mejor_modelo}; el mapa, los escenarios y el ONNX usan {RED}. Hay que decirlo en el sitio.")

## 8. ¿El dato satelital ve los años malos del SIAP? (validación de la etiqueta, no del modelo)
Anomalía regional del NDVI pico observado contra los años malos del maíz en Amealco (SIAP).

In [ ]:
reg_obs = pd.Series(np.nanmean(Y, 1), index=ANIOS)
if not CONFIG["anios_malos_siap"]:
    print("Sin años malos del SIAP para esta zona: la comparación queda en \"sin dato\".")
anom = reg_obs - reg_obs[[a for a in ANIOS if en(CONFIG["train"], a)]].mean()
con_siap = [a for a in anom.index if 2003 <= a <= 2025]          # años que cubre la lista del SIAP
malos = [a for a in CONFIG["anios_malos_siap"] if a in con_siap]
otros = [a for a in con_siap if a not in malos]
print("Anomalía media del NDVI pico (2003-2025) — años malos SIAP:", round(anom[malos].mean(), 4),
      "| otros años:", round(anom[otros].mean(), 4))
print(f"Años malos entre los {len(malos)} más bajos del NDVI pico:",
      sorted(set(malos) & set(anom[con_siap].nsmallest(len(malos)).index)))

## 9. Predicción por píxel para el sitio + escenarios (lluvia y temperatura)

In [ ]:
def predice_anio(anio, factor_lluvia=1.0, delta_t=0.0):
    k = list(ANIOS).index(anio)
    xs = XSEQ[k].copy()
    il, ix_, in_, ie = FEATS.index("lluvia_mm"), FEATS.index("tmax_c"), FEATS.index("tmin_c"), FEATS.index("et0_mm")
    xs[:, :, il] *= factor_lluvia
    if delta_t:
        tm0 = (xs[:, :, ix_] + xs[:, :, in_]) / 2
        xs[:, :, ix_] += delta_t; xs[:, :, in_] += delta_t
        xs[:, :, ie] *= (tm0 + delta_t + 17.8) / (tm0 + 17.8)        # Hargreaves con el mismo rango diario
    xs = (xs - mu_s) / sd_s
    xe = (ESTAT - mu_e) / sd_e
    ok = np.isfinite(xs).all((1, 2)) & np.isfinite(xe).all(1)
    P = np.full((len(modelos), len(iy)), np.nan, "float32")
    for i, m in enumerate(modelos):
        P[i, ok] = predice(m, xs[ok], xe[ok])
    return P.mean(0), np.sqrt(P.std(0) ** 2 + sigma_res ** 2)   # semillas + error de validación (igual que el ONNX)

anio_demo = int(ANIOS.max())
pred, banda = predice_anio(anio_demo)
obs = Y[list(ANIOS).index(anio_demo)]
r3 = lambda a: [None if not np.isfinite(v) else round(float(v), 3) for v in a]

base = {"fecha_consulta": FECHA_CONSULTA, "ejemplo": False, "zona_id": ZONA_ID}
estres = dict(base,
    fuente=f"Predicción {RED} con MOD13A1.061 NDVI + NASA POWER; máscara: {fuente_mascara}",
    anio=anio_demo, variable="NDVI máximo de los compuestos MODIS del 28 jul al 15 oct (predicho con datos hasta el 27 jul)",
    datos={"lat": r3(lats), "lon": r3(lons), "pred": r3(pred), "banda": r3(banda),
           "obs": r3(obs), "clim": r3(CLIM), "anomalia_pred": r3(pred - CLIM)})

escenarios = dict(base, fuente=f"{RED} con entradas abr-jul modificadas (lluvia × factor, temperatura + Δ)", anio=anio_demo, datos=[])
for fl in [0.7, 0.85, 1.0, 1.15]:
    for dt in [0.0, 1.0, 2.0]:
        p, _ = predice_anio(anio_demo, fl, dt)
        escenarios["datos"].append({"lluvia_factor": fl, "delta_t_c": dt,
                                    "anomalia_media": round(float(np.nanmean(p - CLIM)), 4),
                                    "pixeles_estres_pct": round(float(np.mean((p - CLIM)[np.isfinite(p - CLIM)] < -0.05) * 100), 1)})
# antes de mostrar el slider: con menos lluvia de abr-jul la anomalía media no debería subir
an0 = [d["anomalia_media"] for d in escenarios["datos"] if d["delta_t_c"] == 0.0]   # lluvia × 0.7, 0.85, 1.0, 1.15
escenarios["monotono_lluvia"] = bool(all(a <= b + 1e-4 for a, b in zip(an0, an0[1:])))
escenarios["nota"] = ("Sensibilidad del modelo, no pronóstico: solo cambia lluvia y temperatura de abr-jul (lo que el modelo ve); "
                      "la lluvia de ago-sep no entra. El clima es un solo punto de NASA POWER para toda la región. "
                      "Estrés = anomalía de NDVI < -0.05 contra el promedio del píxel (convención del equipo, no umbral agronómico).")
print("Anomalía media con lluvia × 0.7 / 0.85 / 1.0 / 1.15:", an0, "| ¿monótona?", escenarios["monotono_lluvia"])

comparacion = dict(base,
    fuente="Evaluación por año: train %s, val %s, test %s" % (CONFIG["train"], CONFIG["val"], CONFIG["test"]),
    metrica_principal="skill_vs_climatologia (1 - SSE_modelo / SSE_climatología; > 0 = mejor que el promedio histórico)",
    mejor_modelo=mejor_modelo, red_del_sitio=RED, arquitectura_red=FAMILIAS[RED],
    criterio_mejor=f"menor RMSE en validación {CONFIG['val'][0]}-{CONFIG['val'][1]}; la prueba solo se reporta",
    banda_media_lstm=round(banda_test, 4), banda_solo_semillas=round(banda_semillas, 4),
    sigma_residual_val=round(sigma_res, 4), cobertura_banda_prueba=round(cob_tot, 3),
    skill_por_anio_prueba=por_anio, curvas_lstm=CURVAS, variables=FEATS,
    datos=[{"modelo": k, **{f"{s}_{m}": v[s][m] for s in ("val", "test") for m in v[s]}} for k, v in resultados.items()])

serie = dict(base, fuente="MOD13A1.061 NDVI (anomalía regional del pico) y SIAP (años malos del maíz)",
    datos=[{"anio": int(a), "anomalia_ndvi_pico": round(float(anom[a]), 4),
            "anio_malo_siap": ((int(a) in malos) if 2003 <= a <= 2025 else None) if CONFIG["anios_malos_siap"] else None}
           for a in anom.index])   # None = sin dato SIAP

# Confianza en palabras para el agricultor (la pide la mentoría): años de prueba en que la red le ganó al promedio
_gana = sum(1 for v in por_anio[RED].values() if v > 0)
comparacion["confianza_en_palabras"] = (f"En {_gana} de {len(por_anio[RED])} años de prueba ({CONFIG['test'][0]}-{CONFIG['test'][1]}) "
                                        "el modelo se acercó más que el promedio de años pasados.")
zona = dict(base, fuente=f"Lugar del agricultor; terrenos agrícolas MODIS dentro de {RADIO_KM} km",
            datos=[{"zona_id": ZONA_ID, "nombre": NOMBRE_ZONA, "lat": LAT_AGRICULTOR, "lon": LON_AGRICULTOR,
                    "radio_cobertura_km": RADIO_KM, "terrenos_agricolas": int(len(iy)),
                    "radio_cobertura_razon": f"Modelo entrenado solo con terrenos a {RADIO_KM} km o menos de este punto."}])
print(comparacion["confianza_en_palabras"])

def limpia(o):
    """NaN/inf → null y tipos de numpy → Python: el JSON.parse del navegador no acepta NaN."""
    if isinstance(o, dict):
        return {str(k): limpia(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [limpia(v) for v in o]
    if isinstance(o, (float, np.floating)):
        return float(o) if np.isfinite(o) else None
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.bool_):
        return bool(o)
    return o

for nombre, obj in [("estres_mapa.json", estres), ("escenarios.json", escenarios),
                    ("modelos_comparacion.json", comparacion), ("serie_regional.json", serie), ("zona.json", zona)]:
    with open(os.path.join(OUT_DIR, nombre), "w", encoding="utf-8") as fh:
        json.dump(limpia(obj), fh, ensure_ascii=False)
    print("guardado", nombre, os.path.getsize(os.path.join(OUT_DIR, nombre)) // 1024, "KB")

# sin arrays de numpy dentro: así se carga con torch.load(..., weights_only=True)
torch.save({"estados": [m.state_dict() for m in modelos], "feats": FEATS,
            "mu_s": torch.tensor(mu_s), "sd_s": torch.tensor(sd_s), "mu_e": torch.tensor(mu_e), "sd_e": torch.tensor(sd_e),
            "sigma_res": sigma_res, "red": RED, "arquitectura": FAMILIAS[RED], "config": CONFIG}, os.path.join(OUT_DIR, "lstm_fieldshift.pt"))

## 10. Exportar el modelo para que el HTML lo consulte (ONNX, en el navegador)
Un solo archivo `modelo_lstm.onnx` con las 5 semillas: recibe los valores **crudos** (mm, °C, NDVI) y regresa
promedio e incertidumbre (dispersión entre semillas + error de validación, la misma banda de `estres_mapa.json`).
La normalización va dentro del modelo, así el sitio solo cambia lluvia o temperatura y vuelve a preguntar.
Junto va `entradas_<año>.bin` (Float32, secuencia de cada píxel) + `entradas_<año>.json` (forma, orden de variables, lat/lon).

In [ ]:
class Ensamble(nn.Module):
    def __init__(self, redes):
        super().__init__()
        self.redes = nn.ModuleList(redes)
        self.register_buffer("mu_s", torch.tensor(mu_s, dtype=torch.float32))
        self.register_buffer("sd_s", torch.tensor(sd_s, dtype=torch.float32))
        self.register_buffer("mu_e", torch.tensor(mu_e, dtype=torch.float32))
        self.register_buffer("sd_e", torch.tensor(sd_e, dtype=torch.float32))
        self.register_buffer("var_res", torch.tensor(sigma_res ** 2, dtype=torch.float32))
    def forward(self, secuencia, estatico):
        xs = (secuencia - self.mu_s) / self.sd_s
        xe = (estatico - self.mu_e) / self.sd_e
        p = torch.stack([r(xs, xe) for r in self.redes], 0)
        media = p.mean(0)
        # misma banda que estres_mapa.json: de poblacional entre semillas (como numpy) + error de validación
        # las redes dan la anomalía: se suma el promedio del píxel (estatico[:, 0], valor crudo) para regresar NDVI
        return media + estatico[:, 0], torch.sqrt(((p - media) ** 2).mean(0) + self.var_res)

import copy
ens = Ensamble([copy.deepcopy(m).cpu().eval() for m in modelos]).eval()   # copia: los originales siguen en la GPU
k = list(ANIOS).index(anio_demo)
sec = np.nan_to_num(XSEQ[k]).astype("float32"); est = np.nan_to_num(ESTAT).astype("float32")
valido = (np.isfinite(XSEQ[k]).all((1, 2)) & np.isfinite(ESTAT).all(1))
ruta_onnx = os.path.join(OUT_DIR, "modelo_lstm.onnx")
ejemplo = (torch.tensor(sec[:4]), torch.tensor(est[:4]))
kw = dict(input_names=["secuencia", "estatico"], output_names=["ndvi_pico", "incertidumbre"],
          dynamic_axes={"secuencia": {0: "n"}, "estatico": {0: "n"}, "ndvi_pico": {0: "n"}, "incertidumbre": {0: "n"}},
          opset_version=17)
try:
    torch.onnx.export(ens, ejemplo, ruta_onnx, dynamo=False, **kw)   # exportador clásico
except Exception as e:                 # torch sin el exportador clásico o sin el argumento dynamo
    print("Exportador clásico no disponible:", type(e).__name__, "→ uso el exportador nuevo (dynamo)")
    torch.onnx.export(ens, ejemplo, ruta_onnx, dynamo=True, external_data=False, **kw)

# comprobación: ONNX da lo mismo que PyTorch
import onnxruntime as ort
s = ort.InferenceSession(ruta_onnx)
o_mean, o_inc = s.run(None, {"secuencia": sec[valido][:256], "estatico": est[valido][:256]})
with torch.no_grad():
    t_mean, _ = ens(torch.tensor(sec[valido][:256]), torch.tensor(est[valido][:256]))
print("ONNX vs PyTorch, diferencia máxima:", float(np.abs(o_mean - t_mean.numpy()).max()))
# lo que verá el navegador debe coincidir con estres_mapa.json (mismo año, sin escenario)
print("ONNX vs estres_mapa.json · pred:", float(np.abs(o_mean - pred[valido][:256]).max()),
      "| banda:", float(np.abs(o_inc - banda[valido][:256]).max()))

sec.tofile(os.path.join(OUT_DIR, f"entradas_{anio_demo}.bin"))
est.tofile(os.path.join(OUT_DIR, f"estatico_{anio_demo}.bin"))
json.dump(dict(base, fuente="Entradas del modelo: MOD13A1.061 NDVI + NASA POWER (+ SRTM)", anio=anio_demo,
               n=int(len(iy)), pasos=len(CONFIG["doy_entrada"]), variables=FEATS,
               estaticas=["ndvi_clim", "elevacion_m", "pendiente_grados"],
               doy_pasos=CONFIG["doy_entrada"], valido=valido.astype(int).tolist(),
               lat=r3(lats), lon=r3(lons), clim=r3(CLIM)),
          open(os.path.join(OUT_DIR, f"entradas_{anio_demo}.json"), "w", encoding="utf-8"), ensure_ascii=False)
for f in sorted(os.listdir(OUT_DIR)):
    print(f, os.path.getsize(os.path.join(OUT_DIR, f)) // 1024, "KB")